# Paso 6: Validación de Datos Sintéticos vs. Literatura

---

## ¿Para qué sirve este notebook?

Los datos sintéticos son útiles para desarrollar el sistema, pero deben ser realistas. En este notebook comparamos nuestras distribuciones con valores publicados en la literatura científica. El objetivo es responder a una pregunta clave: **¿son los datos que hemos generado clínicamente plausibles para una población de deportistas?**

Si los datos sintéticos tienen distribuciones absurdas (por ejemplo, cuádriceps de 10 N o dorsiflexión de 120°), el modelo aprenderá patrones que no existen en la realidad, y el sistema no tendrá validez metodológica.

---

## Marco de interpretación (importante para el TFM)

Este proyecto es una **demostración de viabilidad metodológica**. La validación clínica real requiere un estudio prospectivo con datos de cohorte: pacientes reales, seguimiento en el tiempo, y verificación de si las predicciones del modelo se cumplen.

Lo que hacemos aquí es más modesto pero igualmente riguroso: **demostramos que los datos sintéticos usados para entrenar el modelo se encuentran dentro de los rangos reportados en la literatura** para poblaciones deportivas similares. Esto legitima el desarrollo técnico del sistema aunque no constituya validación clínica.

---

## Estructura de este notebook

| Sección | Contenido |
|---------|----------|
| 2 | Carga de librerías y datos |
| 3 | Tabla de referencia bibliográfica (requiere tu trabajo) |
| 4 | Comparación visual distribuciones vs. literatura |
| 5 | Test de normalidad (Shapiro-Wilk) |
| 6 | Correlaciones clínicas esperadas |
| 7 | Diagnóstico automático de realismo |
| 8 | **Tu trabajo**: instrucciones para completar la validación |

## Sección 2: Configuración y carga de datos

In [1]:
import sys
from pathlib import Path

# Añadir la raíz del proyecto al path para importar módulos propios
proyecto_raiz = Path("..").resolve()
sys.path.insert(0, str(proyecto_raiz))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib
from scipy import stats
import warnings
warnings.filterwarnings("ignore")

matplotlib.rcParams["font.family"] = "DejaVu Sans"
matplotlib.rcParams["figure.dpi"] = 120

# Importar definición central de variables
from src.variables import VARIABLES, VARIABLES_ORIGINALES

print(f"Variables cargadas desde src/variables.py: {len(VARIABLES)} columnas expandidas")

Variables cargadas desde src/variables.py: 33 columnas expandidas


In [2]:
# Cargar los datos sintéticos generados en el notebook 01
ruta_csv = proyecto_raiz / "datos" / "sinteticos" / "dataset_sintetico.csv"

if not ruta_csv.exists():
    raise FileNotFoundError(
        f"No se encuentra {ruta_csv}. "
        "Ejecuta primero el notebook 01_generacion_datos.ipynb."
    )

df = pd.read_csv(ruta_csv)
print(f"Dataset cargado: {df.shape[0]} deportistas, {df.shape[1]} columnas")
print(f"\nPrimeras 3 filas:")
df.head(3)

Dataset cargado: 500 deportistas, 38 columnas

Primeras 3 filas:


,cuadriceps_der,cuadriceps_izq,isquiotibiales_der,isquiotibiales_izq,gluteo_medio_der,gluteo_medio_izq,rotadores_externos_cadera_der,rotadores_externos_cadera_izq,aductores_cadera_der,aductores_cadera_izq,...,perfil_exigencia_deportiva,historial_lesional,dolor_percibido_nrs,acwr,indice_estres_descanso,riesgo_lesion,score_total,confianza_score,confianza_categoria,reglas_activadas
0,289.7,383.2,165.3,180.5,120.5,109.1,143.9,133.1,194.9,187.4,...,3,2,4,1.2,18.7,bajo,4.0,92.6,alta,—
1,349.4,320.2,232.6,268.0,140.3,138.5,135.3,128.2,191.3,221.6,...,4,1,2,1.1,17.5,bajo,15.5,66.7,media,—
2,175.5,249.4,142.7,151.7,74.9,56.2,90.0,64.8,83.1,111.5,...,3,2,0,1.0,15.7,medio,19.0,63.0,media,—


In [3]:
# Estadísticas descriptivas de las variables numéricas
# Guardamos esto para usarlo en la tabla comparativa de la Sección 3
stats_sinteticos = df.select_dtypes(include=[np.number]).agg(["mean", "std", "min", "max"]).T
stats_sinteticos.columns = ["media_sintetica", "de_sintetica", "min_sintetico", "max_sintetico"]
stats_sinteticos = stats_sinteticos.round(2)

print(f"Estadísticas calculadas para {len(stats_sinteticos)} variables numéricas.")
stats_sinteticos.head(10)

Estadísticas calculadas para 33 variables numéricas.


,media_sintetica,de_sintetica,min_sintetico,max_sintetico
cuadriceps_der,353.95,127.39,80.0,600.0
cuadriceps_izq,356.79,125.92,80.0,600.0
isquiotibiales_der,220.59,83.75,50.0,400.0
isquiotibiales_izq,225.95,81.86,50.0,400.0
gluteo_medio_der,177.34,65.70,40.0,320.0
gluteo_medio_izq,177.23,65.33,40.0,320.0
rotadores_externos_cadera_der,125.68,49.21,25.0,230.0
rotadores_externos_cadera_izq,123.76,49.16,25.5,230.0
aductores_cadera_der,190.74,67.48,50.0,340.0
aductores_cadera_izq,193.00,68.40,50.0,340.0


## Sección 3: Tabla de referencia bibliográfica

### Instrucciones para Roberto

A continuación encontrarás un diccionario con marcadores `TODO ROBERTO`. **Tu tarea es buscar en la literatura científica los valores de referencia para cada variable y rellenarlos aquí.**

Para cada variable necesitas:
1. La **media** publicada en un estudio con población similar a la tuya (deportistas, edad aproximada, género)
2. La **desviación estándar** (o el rango ± 1 SD) de ese mismo estudio
3. La **referencia completa** del artículo (autor, año, revista)
4. La **población** del estudio (para poder comparar con la tuya)

Fuentes recomendadas: PubMed, SPORTDiscus, Google Scholar. Términos de búsqueda: `[variable] normative values athletes`, `[variable] reference values footballers`.

**Si no encuentras un valor**: deja `None` y documenta en el TFM que esa variable carece de referencia normativa publicada para tu población.

In [4]:
# =============================================================================
# Tabla de referencias bibliográficas — solo variables del dataset real.
# Unidades: N (fuerza HHD), cm (WBLT), ° (movilidad angular), % (Y-Balance CS)
#           repeticiones (tríceps sural).
# [ROBERTO] = requiere búsqueda adicional en PubMed/SPORTDiscus.
# =============================================================================

referencias = {

    # --- BLOQUE FUERZA (N) -----------------------------------------------
    # Nota: literatura típicamente en N/kg; aquí N/kg × peso_medio 71 kg.

    'cuadriceps_der': {
        'media_literatura': 389.0,   # 5.48 N/kg × 71 kg (fútbol ♂)
        'de_literatura':    70.0,
        'referencia': 'Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.',
        'poblacion':  'Futbolistas/baloncestistas universitarios ♂/♀, 18-30 años.',
    },
    'cuadriceps_izq': {
        'media_literatura': 389.0,
        'de_literatura':    70.0,
        'referencia': 'Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.',
        'poblacion':  'Futbolistas/baloncestistas universitarios ♂/♀, 18-30 años.',
    },
    'isquiotibiales_der': {
        'media_literatura': 234.0,   # 3.29 N/kg × 71 kg (fútbol ♂)
        'de_literatura':    50.0,
        'referencia': 'Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.',
        'poblacion':  'Futbolistas universitarios ♂, 18-30 años.',
    },
    'isquiotibiales_izq': {
        'media_literatura': 234.0,
        'de_literatura':    50.0,
        'referencia': 'Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.',
        'poblacion':  'Futbolistas universitarios ♂, 18-30 años.',
    },
    'gluteo_medio_der': {
        'media_literatura': None,    # [ROBERTO] Thorborg (2016): 1.41 Nm/kg — convertir a N
        'de_literatura':    None,
        'referencia': 'Thorborg et al. (2016). Br J Sports Med 50(12):e2.',
        'poblacion':  '[ROBERTO: n, edad, deporte del estudio]',
    },
    'gluteo_medio_izq': {
        'media_literatura': None,
        'de_literatura':    None,
        'referencia': 'Thorborg et al. (2016). Br J Sports Med 50(12):e2.',
        'poblacion':  '[ROBERTO]',
    },
    'rotadores_externos_cadera_der': {
        'media_literatura': None,    # [ROBERTO] ~0.46 Nm/kg (PMC6442714, 2019) — convertir a N
        'de_literatura':    None,
        'referencia': '[ROBERTO: Thorborg et al. (2012) o similar]',
        'poblacion':  '[ROBERTO]',
    },
    'rotadores_externos_cadera_izq': {
        'media_literatura': None,
        'de_literatura':    None,
        'referencia': '[ROBERTO]',
        'poblacion':  '[ROBERTO]',
    },
    'aductores_cadera_der': {
        'media_literatura': None,    # [ROBERTO] ~1.85-2.40 Nm/kg — convertir a N
        'de_literatura':    None,
        'referencia': 'Thorborg et al. (2011). Br J Sports Med 45(6):501-508.',
        'poblacion':  '[ROBERTO]',
    },
    'aductores_cadera_izq': {
        'media_literatura': None,
        'de_literatura':    None,
        'referencia': '[ROBERTO]',
        'poblacion':  '[ROBERTO]',
    },
    'triceps_sural_der': {
        'media_literatura': 24.0,    # ♂ ~24 rep, ♀ ~21 rep
        'de_literatura':    4.5,
        'referencia': 'Hébert-Losier et al. (2017). Sports Med 47(3):517-554.',
        'poblacion':  'Adultos sanos 18-65 años. ♂: 24 rep (±4.5), ♀: 21 rep (±4.1).',
    },
    'triceps_sural_izq': {
        'media_literatura': 24.0,
        'de_literatura':    4.5,
        'referencia': 'Hébert-Losier et al. (2017). Sports Med 47(3):517-554.',
        'poblacion':  'Adultos sanos 18-65 años.',
    },

    # --- BLOQUE MOVILIDAD -----------------------------------------------

    'dorsiflexion_tobillo_der': {
        'media_literatura': 13.7,    # ♂ ~14.3, ♀ ~13.0 → media mixta ~13.7 cm
        'de_literatura':    2.5,
        'referencia': 'Powden et al. (2015). Int J Sports Phys Ther 10(1):21-29.',
        'poblacion':  'Adultos activos 18-45 años, WBLT (cm).',
    },
    'dorsiflexion_tobillo_izq': {
        'media_literatura': 13.7,
        'de_literatura':    2.5,
        'referencia': 'Powden et al. (2015). Int J Sports Phys Ther 10(1):21-29.',
        'poblacion':  'Adultos activos 18-45 años.',
    },
    'thomas_test_der': {
        'media_literatura': None,    # Variable binaria (0=negativo, 1=positivo) — no aplica media
        'de_literatura':    None,
        'referencia': 'Harvey et al. (1998). Physiotherapy 84(4):173-175.',
        'poblacion':  '[ROBERTO: prevalencia positivo en población objetivo]',
    },
    'thomas_test_izq': {
        'media_literatura': None,
        'de_literatura':    None,
        'referencia': 'Harvey et al. (1998). Physiotherapy 84(4):173-175.',
        'poblacion':  '[ROBERTO]',
    },
    'rotacion_interna_cadera_der': {
        'media_literatura': 33.4,    # Media mixta: ♂ 28.6, ♀ 38.1 (Teng 2018)
        'de_literatura':    9.0,
        'referencia': 'Teng et al. (2018). Phys Ther Sport 31:60-65.',
        'poblacion':  'Deportistas 18-40 años. ♂: 28.6±8.4°, ♀: 38.1±8.2°.',
    },
    'rotacion_interna_cadera_izq': {
        'media_literatura': 33.4,
        'de_literatura':    9.0,
        'referencia': 'Teng et al. (2018). Phys Ther Sport 31:60-65.',
        'poblacion':  'Deportistas 18-40 años.',
    },

    # --- BLOQUE CONTROL Y EQUILIBRIO ------------------------------------

    'y_balance_cs_der': {
        'media_literatura': 94.0,    # Composite Score %; umbral lesión < 94%
        'de_literatura':    6.0,
        'referencia': 'Plisky et al. (2006). N Am J Sports Phys Ther 1(3):92-99.',
        'poblacion':  'Deportistas ♀ secundaria. [ROBERTO: ref ♂ adicional].',
    },
    'y_balance_cs_izq': {
        'media_literatura': 94.0,
        'de_literatura':    6.0,
        'referencia': 'Plisky et al. (2006). N Am J Sports Phys Ther 1(3):92-99.',
        'poblacion':  'Deportistas ♀ secundaria.',
    },
    'single_leg_squat_valgo_der': {
        'media_literatura': None,    # Ordinal 0-3 — no aplica media numérica directa
        'de_literatura':    None,
        'referencia': 'Crossley et al. (2011). Br J Sports Med 45(6):500.',
        'poblacion':  '[ROBERTO: prevalencia valgo ≥1 en población objetivo]',
    },
    'single_leg_squat_valgo_izq': {
        'media_literatura': None,
        'de_literatura':    None,
        'referencia': 'Crossley et al. (2011). Br J Sports Med 45(6):500.',
        'poblacion':  '[ROBERTO]',
    },
    'single_leg_hop_der': {
        'media_literatura': 155.0,   # Media mixta deportistas sanos 18-40 años (cm)
        'de_literatura':    25.0,
        'referencia': 'Reid et al. (2007). JOSPT 37(2):57-64.',
        'poblacion':  'Adultos activos 18-40 años. ♂: ~165 cm, ♀: ~130 cm.',
    },
    'single_leg_hop_izq': {
        'media_literatura': 155.0,
        'de_literatura':    25.0,
        'referencia': 'Reid et al. (2007). JOSPT 37(2):57-64.',
        'poblacion':  'Adultos activos 18-40 años.',
    },

    # --- BLOQUE CONTEXTO ------------------------------------------------

    'edad': {
        'media_literatura': 26.0,
        'de_literatura':    5.0,
        'referencia': 'Caracterización de la población objetivo del TFM.',
        'poblacion':  'Deportistas activos 18-35 años predominantemente.',
    },
    'peso_corporal': {
        'media_literatura': 72.0,
        'de_literatura':    12.0,
        'referencia': 'Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.',
        'poblacion':  'Deportistas universitarios mixtos 18-30 años.',
    },
    'dolor_percibido_nrs': {
        'media_literatura': 1.5,
        'de_literatura':    1.8,
        'referencia': 'Williamson & Hoggart (2005). J Clin Nurs 14(7):798-804.',
        'poblacion':  'Deportistas sin lesión aguda (dolor basal 0-3/10).',
    },
    'acwr': {
        'media_literatura': 1.0,
        'de_literatura':    0.2,
        'referencia': 'Hulin et al. (2016). Br J Sports Med 50(4):231-236.',
        'poblacion':  'Deportistas en período competitivo. Zona segura: 0.8-1.3.',
    },
    # Variables categóricas/ordinales — sin comparación numérica directa
    'genero':                    {'media_literatura': None, 'de_literatura': None,
                                  'referencia': 'Variable categórica (0=♂, 1=♀).', 'poblacion': 'Distribución aprox. uniforme en dataset.'},
    'nivel_actividad':           {'media_literatura': None, 'de_literatura': None,
                                  'referencia': 'Variable ordinal 0=sedentario…3=élite.', 'poblacion': '[ROBERTO]'},
    'perfil_exigencia_deportiva':{'media_literatura': None, 'de_literatura': None,
                                  'referencia': 'Variable ordinal 0-4.', 'poblacion': '[ROBERTO]'},
    'historial_lesional':        {'media_literatura': None, 'de_literatura': None,
                                  'referencia': 'Variable binaria (0=sin historial, 1=con historial).', 'poblacion': '[ROBERTO: prevalencia lesión previa]'},
    'indice_estres_descanso':    {'media_literatura': None, 'de_literatura': None,
                                  'referencia': '[ROBERTO: describir índice y rango normal]', 'poblacion': '[ROBERTO]'},
}

print(f'Tabla de referencias: {len(referencias)} variables.')
rellenas = sum(1 for r in referencias.values() if r['media_literatura'] is not None)
print(f'  Con valor pre-rellenado: {rellenas}')
print(f'  Pendientes [ROBERTO]:    {len(referencias) - rellenas}')


Tabla de referencias: 33 variables.
  Con valor pre-rellenado: 18
  Pendientes [ROBERTO]:    15


In [5]:
# Construir la tabla comparativa completa, combinando referencias con estadísticas sintéticas
filas = []
for var, ref in referencias.items():
    if var not in stats_sinteticos.index:
        continue
    media_sint = stats_sinteticos.loc[var, "media_sintetica"]
    de_sint    = stats_sinteticos.loc[var, "de_sintetica"]
    media_lit  = ref["media_literatura"]
    de_lit     = ref["de_literatura"]

    # Diferencia porcentual (solo si tenemos ambos valores)
    if media_lit is not None and media_lit != 0:
        dif_pct = round(abs(media_sint - media_lit) / media_lit * 100, 1)
    else:
        dif_pct = None

    filas.append({
        "variable":        var,
        "media_sintetica": media_sint,
        "de_sintetica":    de_sint,
        "media_literatura":media_lit if media_lit is not None else "pendiente",
        "de_literatura":   de_lit    if de_lit    is not None else "pendiente",
        "diferencia_pct":  f"{dif_pct}%" if dif_pct is not None else "–",
        "referencia":      ref["referencia"],
        "poblacion":       ref["poblacion"],
    })

df_comparacion = pd.DataFrame(filas)
print("Tabla de comparación (primeras 10 filas):")
df_comparacion[["variable", "media_sintetica", "de_sintetica",
                "media_literatura", "de_literatura", "diferencia_pct"]].head(10)

Tabla de comparación (primeras 10 filas):


,variable,media_sintetica,de_sintetica,media_literatura,de_literatura,diferencia_pct
0,cuadriceps_der,353.95,127.39,389.0,70.0,9.0%
1,cuadriceps_izq,356.79,125.92,389.0,70.0,8.3%
2,isquiotibiales_der,220.59,83.75,234.0,50.0,5.7%
3,isquiotibiales_izq,225.95,81.86,234.0,50.0,3.4%
4,gluteo_medio_der,177.34,65.70,pendiente,pendiente,–
5,gluteo_medio_izq,177.23,65.33,pendiente,pendiente,–
6,rotadores_externos_cadera_der,125.68,49.21,pendiente,pendiente,–
7,rotadores_externos_cadera_izq,123.76,49.16,pendiente,pendiente,–
8,aductores_cadera_der,190.74,67.48,pendiente,pendiente,–
9,aductores_cadera_izq,193.00,68.40,pendiente,pendiente,–


In [6]:
# Tabla completa con referencias bibliográficas
print("Tabla completa con referencias:")
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 60)
df_comparacion[["variable", "media_sintetica", "media_literatura",
                "diferencia_pct", "referencia"]]

Tabla completa con referencias:


,variable,media_sintetica,media_literatura,diferencia_pct,referencia
0,cuadriceps_der,353.95,389.0,9.0%,Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.
1,cuadriceps_izq,356.79,389.0,8.3%,Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.
2,isquiotibiales_der,220.59,234.0,5.7%,Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.
3,isquiotibiales_izq,225.95,234.0,3.4%,Owoeye et al. (2024). Int J Exerc Sci 17(4):768-778.
4,gluteo_medio_der,177.34,pendiente,–,Thorborg et al. (2016). Br J Sports Med 50(12):e2.
5,gluteo_medio_izq,177.23,pendiente,–,Thorborg et al. (2016). Br J Sports Med 50(12):e2.
6,rotadores_externos_cadera_der,125.68,pendiente,–,[ROBERTO: Thorborg et al. (2012) o similar]
7,rotadores_externos_cadera_izq,123.76,pendiente,–,[ROBERTO]
8,aductores_cadera_der,190.74,pendiente,–,Thorborg et al. (2011). Br J Sports Med 45(6):501-508.
9,aductores_cadera_izq,193.00,pendiente,–,[ROBERTO]


## Sección 4: Comparación visual — Distribuciones sintéticas vs. literatura

Para cada variable clave se muestra el histograma de los datos sintéticos. Cuando hayas rellenado los valores de la literatura, aparecerán líneas verticales indicando la media ± 1 SD del estudio de referencia.

- **Línea sólida naranja**: media publicada en la literatura
- **Área sombreada naranja**: ± 1 desviación estándar de la literatura
- **Barras azules**: distribución de los datos sintéticos

Si la línea naranja cae bien dentro del histograma azul, la distribución es coherente. Si queda muy a la derecha o a la izquierda, los parámetros en `src/variables.py` deben ajustarse.

In [7]:
VARS_VISUALIZACION = [
    'cuadriceps_der',
    'isquiotibiales_der',
    'gluteo_medio_der',
    'triceps_sural_der',
    'dorsiflexion_tobillo_der',
    'rotacion_interna_cadera_der',
    'y_balance_cs_der',
    'single_leg_hop_der',
]
VARS_VISUALIZACION = [v for v in VARS_VISUALIZACION if v in df.columns]
print(f'Variables para visualización ({len(VARS_VISUALIZACION)}): {VARS_VISUALIZACION}')


Variables para visualización (8): ['cuadriceps_der', 'isquiotibiales_der', 'gluteo_medio_der', 'triceps_sural_der', 'dorsiflexion_tobillo_der', 'rotacion_interna_cadera_der', 'y_balance_cs_der', 'single_leg_hop_der']


In [8]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for idx, var in enumerate(VARS_VISUALIZACION):
    ax = axes[idx]
    datos = df[var].dropna()

    # Histograma de datos sintéticos
    ax.hist(datos, bins=30, color="#3498db", alpha=0.75, edgecolor="white",
            linewidth=0.5, label="Datos sintéticos", density=True)

    # Recuperar valores de literatura si existen
    ref = referencias.get(var, {})
    media_lit = ref.get("media_literatura")
    de_lit    = ref.get("de_literatura")

    if media_lit is not None:
        ax.axvline(media_lit, color="#e67e22", linewidth=2.0, linestyle="-",
                   label=f"Media lit. = {media_lit}")
        if de_lit is not None:
            ax.axvspan(media_lit - de_lit, media_lit + de_lit,
                       alpha=0.20, color="#e67e22", label="±1 SD lit.")
        ax.legend(fontsize=7, loc="upper right")
    else:
        ax.text(0.97, 0.95, "Valor literaturapendiente",
                transform=ax.transAxes, fontsize=7, ha="right", va="top",
                color="#c0392b",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#fadbd8", alpha=0.8))

    # Línea de la media sintética
    ax.axvline(datos.mean(), color="#2980b9", linewidth=1.5, linestyle="--",
               label=f"Media sint. = {datos.mean():.1f}")

    # Etiquetas
    info_var = VARIABLES.get(var, {})
    nombre   = info_var.get("nombre_display", var)
    unidad   = info_var.get("unidad", "")
    ax.set_title(nombre, fontsize=9, fontweight="bold", pad=6)
    ax.set_xlabel(unidad if unidad else var, fontsize=8)
    ax.set_ylabel("Densidad", fontsize=8)
    ax.tick_params(labelsize=8)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

# Ocultar subplots vacíos si hay menos de 8 variables
for idx in range(len(VARS_VISUALIZACION), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle(
    "Comparación: distribuciones sintéticas vs. valores de referencia (literatura)",
    fontsize=13, fontweight="bold", y=1.02
)
plt.tight_layout()

ruta_fig1 = proyecto_raiz / "figuras" / "validacion_vs_literatura.png"
plt.savefig(ruta_fig1, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {ruta_fig1}")
plt.close(fig)

Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/validacion_vs_literatura.png


## Sección 5: Test estadístico de normalidad (Shapiro-Wilk)

El test de Shapiro-Wilk contrasta si una variable sigue una distribución normal. Muchos modelos estadísticos asumen normalidad en los datos; también es útil saber si nuestras variables sintéticas tienen la forma de distribución esperada clínicamente.

- **p > 0.05**: no hay evidencia suficiente para rechazar la normalidad (la variable *puede* seguir una distribución normal)
- **p <= 0.05**: la distribución se aparta significativamente de la normal

En variables clínicas no siempre esperamos normalidad perfecta. Por ejemplo, el valgo dinámico (single-leg squat) es una escala ordinal 0-3, claramente no normal. Lo importante es que la forma de la distribución tenga sentido clínico.

> **Nota para el TFM**: si una variable muestra no-normalidad, esto no invalida el análisis. Simplemente documenta que se han utilizado métodos robustos a la distribución (Random Forest, Gradient Boosting) que no asumen normalidad.

In [9]:
# Test de Shapiro-Wilk para variables numéricas continuas
# (se excluyen variables categóricas y ordinales de escala reducida)
EXCLUIR_NORMALIDAD = {
    "single_leg_squat_valgo_der",
    "single_leg_squat_valgo_izq",
    "historial_lesional",
    "dolor_percibido_nrs",
    "perfil_exigencia_deportiva",
    "genero",
    "nivel_riesgo",
}

resultados_normalidad = []
cols_numericas = df.select_dtypes(include=[np.number]).columns.tolist()

for col in cols_numericas:
    if col in EXCLUIR_NORMALIDAD:
        continue
    datos_col = df[col].dropna()
    if len(datos_col) < 3:
        continue
    # Shapiro-Wilk requiere n <= 5000; si hay más, submuestreamos
    muestra = datos_col.sample(min(5000, len(datos_col)), random_state=42)
    stat, p_valor = stats.shapiro(muestra)
    resultados_normalidad.append({
        "variable": col,
        "n": len(datos_col),
        "estadístico_W": round(stat, 4),
        "p_valor": round(p_valor, 4),
        "normal_p005": "Sí" if p_valor > 0.05 else "No",
    })

df_normalidad = pd.DataFrame(resultados_normalidad).sort_values("p_valor")
n_normal     = (df_normalidad["normal_p005"] == "Sí").sum()
n_no_normal  = (df_normalidad["normal_p005"] == "No").sum()

print(f"Variables testadas: {len(df_normalidad)}")
print(f"  Distribución compatible con normal (p > 0.05): {n_normal}")
print(f"  Distribución no normal (p <= 0.05):            {n_no_normal}")
print()
df_normalidad

Variables testadas: 28
  Distribución compatible con normal (p > 0.05): 10
  Distribución no normal (p <= 0.05):            18



,variable,n,estadístico_W,p_valor,normal_p005
0,cuadriceps_der,500,0.9732,0.0000,No
24,acwr,500,0.9512,0.0000,No
22,edad,500,0.9617,0.0000,No
15,thomas_test_izq,500,0.3788,0.0000,No
14,thomas_test_der,500,0.3889,0.0000,No
26,score_total,500,0.9569,0.0000,No
9,aductores_cadera_izq,500,0.9687,0.0000,No
8,aductores_cadera_der,500,0.9715,0.0000,No
27,confianza_score,500,0.9713,0.0000,No
6,rotadores_externos_cadera_der,500,0.9586,0.0000,No


In [10]:
# Resumen visual: Q-Q plots para 6 variables representativas
VARS_QQ = [v for v in VARS_VISUALIZACION if v in df.select_dtypes(include=[np.number]).columns
           and v not in EXCLUIR_NORMALIDAD][:6]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for idx, var in enumerate(VARS_QQ):
    ax = axes[idx]
    datos_col = df[var].dropna()
    stats.probplot(datos_col, dist="norm", plot=ax)
    ax.set_title(VARIABLES.get(var, {}).get("nombre_display", var), fontsize=9, fontweight="bold")
    ax.get_lines()[0].set(markersize=2, alpha=0.5, color="#3498db")
    ax.get_lines()[1].set(color="#e74c3c", linewidth=1.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

for idx in range(len(VARS_QQ), len(axes)):
    axes[idx].set_visible(False)

fig.suptitle("Q-Q plots: ¿se ajustan las variables a una distribución normal?",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.close(fig)

print("\nInterpretación: si los puntos azules siguen la línea roja,"
      " la variable se comporta de forma aproximadamente normal.")


Interpretación: si los puntos azules siguen la línea roja, la variable se comporta de forma aproximadamente normal.


## Sección 6: Análisis de correlaciones clínicas

Más allá de las distribuciones marginales, los datos sintéticos deben reflejar las **relaciones entre variables** documentadas en la literatura. Si generamos cuádriceps y salto de forma completamente independiente, el modelo aprenderá relaciones espurias.

Aquí verificamos tres correlaciones clínicas con base científica:

1. **Glúteo medio (fuerza) ↔ Valgo dinámico (single-leg squat)**: correlación negativa. A más fuerza de glúteo, menos valgo. Respaldado por Powers (2010).
2. **Cuádriceps (fuerza) ↔ Single-leg hop (distancia)**: correlación positiva. Más fuerza = más distancia en el salto. Relación biomecanicamente obvia.
3. **Edad ↔ Flexibilidad (extensibilidad isquiotibial)**: correlación negativa. Con la edad, la flexibilidad tiende a disminuir.

In [11]:
correlaciones_clinicas = [
    {
        'var_x': 'gluteo_medio_der',
        'var_y': 'single_leg_squat_valgo_der',
        'direccion_esperada': 'negativa',
        'hipotesis': 'A mayor fuerza de glúteo medio → menor valgo dinámico',
        'referencia': 'Powers (2010). J Orthop Sports Phys Ther 40(2):42-51.',
    },
    {
        'var_x': 'cuadriceps_der',
        'var_y': 'single_leg_hop_der',
        'direccion_esperada': 'positiva',
        'hipotesis': 'A mayor fuerza de cuádriceps → mayor distancia en SLH',
        'referencia': 'Reid et al. (2007). JOSPT 37(2):57-64.',
    },
    {
        'var_x': 'edad',
        'var_y': 'dorsiflexion_tobillo_der',
        'direccion_esperada': 'negativa',
        'hipotesis': 'A mayor edad → menor dorsiflexión de tobillo',
        'referencia': 'Johanson et al. (2009). Foot Ankle Int 30(12):1173-1178.',
    },
]
correlaciones_validas = [
    c for c in correlaciones_clinicas
    if c['var_x'] in df.columns and c['var_y'] in df.columns
]
print(f'Correlaciones clínicas a verificar: {len(correlaciones_validas)}')


Correlaciones clínicas a verificar: 3


In [12]:
fig, axes = plt.subplots(1, len(correlaciones_validas),
                          figsize=(6 * len(correlaciones_validas), 5))
if len(correlaciones_validas) == 1:
    axes = [axes]

resultados_corr = []

for ax, corr in zip(axes, correlaciones_validas):
    x = df[corr["var_x"]].dropna()
    y = df[corr["var_y"]].dropna()

    # Alinear índices
    idx_comun = x.index.intersection(y.index)
    x, y = x.loc[idx_comun], y.loc[idx_comun]

    # Calcular correlación de Spearman (robusta a no-normalidad)
    r_spearman, p_spearman = stats.spearmanr(x, y)
    r_pearson,  p_pearson  = stats.pearsonr(x, y)

    # Verificar dirección
    dir_observada = "positiva" if r_spearman > 0 else "negativa"
    coincide      = dir_observada == corr["direccion_esperada"]

    resultados_corr.append({
        "hipótesis":          corr["hipotesis"],
        "var_x":              corr["var_x"],
        "var_y":              corr["var_y"],
        "r_Spearman":         round(r_spearman, 3),
        "p_valor":            round(p_spearman, 4),
        "dirección_esperada": corr["direccion_esperada"],
        "dirección_observada":dir_observada,
        "coincide":           "Sí" if coincide else "NO",
        "referencia":         corr["referencia"],
    })

    # Scatter plot con línea de tendencia
    color = "#27ae60" if coincide else "#c0392b"
    ax.scatter(x, y, alpha=0.3, s=12, color="#3498db")

    # Línea de regresión
    m, b = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 200)
    ax.plot(x_line, m * x_line + b, color=color, linewidth=2)

    nombre_x = VARIABLES.get(corr["var_x"], {}).get("nombre_display", corr["var_x"])
    nombre_y = VARIABLES.get(corr["var_y"], {}).get("nombre_display", corr["var_y"])
    ax.set_xlabel(nombre_x, fontsize=9)
    ax.set_ylabel(nombre_y, fontsize=9)

    estado = "CONFIRMA" if coincide else "CONTRADICE"
    ax.set_title(
        f"{estado} hipótesis clínica\nr = {r_spearman:.3f} | p = {p_spearman:.4f}",
        fontsize=9, fontweight="bold",
        color="#27ae60" if coincide else "#c0392b"
    )
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

fig.suptitle("Verificación de correlaciones clínicas en datos sintéticos",
             fontsize=12, fontweight="bold")
plt.tight_layout()

ruta_fig2 = proyecto_raiz / "figuras" / "correlaciones_clinicas.png"
plt.savefig(ruta_fig2, dpi=150, bbox_inches="tight")
print(f"Figura guardada en: {ruta_fig2}")
plt.close(fig)

Figura guardada en: /Users/__robeerr/Programacion_Local/IntApp v2/figuras/correlaciones_clinicas.png


In [13]:
# Tabla resumen de resultados de correlaciones clínicas
df_corr_resultados = pd.DataFrame(resultados_corr)
print("Resumen de verificación de correlaciones clínicas:")
df_corr_resultados[[
    "hipótesis", "r_Spearman", "p_valor",
    "dirección_esperada", "dirección_observada", "coincide"
]]

Resumen de verificación de correlaciones clínicas:


,hipótesis,r_Spearman,p_valor,dirección_esperada,dirección_observada,coincide
0,A mayor fuerza de glúteo medio → menor valgo dinámico,-0.070,0.1158,negativa,negativa,Sí
1,A mayor fuerza de cuádriceps → mayor distancia en SLH,-0.073,0.1036,positiva,negativa,NO
2,A mayor edad → menor dorsiflexión de tobillo,0.029,0.5132,negativa,positiva,NO


## Sección 7: Diagnóstico automático de realismo

Para cada variable numérica, comparamos automáticamente los valores sintéticos con los rangos normales definidos en `src/variables.py`. Este diagnóstico **no sustituye la comparación con la literatura** (Sección 3), pero detecta problemas obvios: por ejemplo, si el 95% de los valores generados cae fuera del rango clínico considerado normal.

**Criterio de clasificación**:
- **Realista**: ≥ 60% de los valores se encuentran dentro del `rango_normal` definido en `variables.py`
- **Revisar**: entre 40% y 60% dentro del rango normal
- **Ajustar**: < 40% dentro del rango normal

In [14]:
diagnostico_filas = []

for var_nombre, info in VARIABLES.items():
    if var_nombre not in df.columns:
        continue
    if info.get("tipo") not in ("continua",):
        continue
    rango = info.get("rango_normal")
    if rango is None:
        continue

    datos_var = df[var_nombre].dropna()
    if len(datos_var) == 0:
        continue

    lo, hi = rango
    pct_dentro = ((datos_var >= lo) & (datos_var <= hi)).mean() * 100

    if pct_dentro >= 60:
        estado = "Realista"
    elif pct_dentro >= 40:
        estado = "Revisar"
    else:
        estado = "Ajustar"

    diagnostico_filas.append({
        "variable":          var_nombre,
        "nombre_display":    info.get("nombre_display", var_nombre),
        "bloque":            info.get("bloque", ""),
        "unidad":            info.get("unidad", ""),
        "rango_normal":      f"[{lo}, {hi}]",
        "media_sintetica":   round(datos_var.mean(), 2),
        "pct_dentro_rango":  round(pct_dentro, 1),
        "estado":            estado,
    })

df_diagnostico = pd.DataFrame(diagnostico_filas).sort_values(["estado", "bloque"])

n_realista = (df_diagnostico["estado"] == "Realista").sum()
n_revisar  = (df_diagnostico["estado"] == "Revisar").sum()
n_ajustar  = (df_diagnostico["estado"] == "Ajustar").sum()

print("Diagnóstico de realismo:")
print(f"  Realista : {n_realista} variables")
print(f"  Revisar  : {n_revisar} variables")
print(f"  Ajustar  : {n_ajustar} variables")

Diagnóstico de realismo:
  Realista : 23 variables
  Revisar  : 1 variables
  Ajustar  : 0 variables


In [15]:
# Mostrar tabla de diagnóstico con colores
def colorear_estado(val):
    colores_estado = {
        "Realista": "background-color: #d5f5e3; color: #1e8449",
        "Revisar":  "background-color: #fef9e7; color: #d68910",
        "Ajustar":  "background-color: #fadbd8; color: #c0392b",
    }
    return colores_estado.get(val, "")

df_diagnostico_display = df_diagnostico[[
    "nombre_display", "bloque", "unidad",
    "rango_normal", "media_sintetica", "pct_dentro_rango", "estado"
]].rename(columns={
    "nombre_display":   "Variable",
    "bloque":           "Bloque",
    "unidad":           "Unidad",
    "rango_normal":     "Rango normal",
    "media_sintetica":  "Media sintética",
    "pct_dentro_rango": "% dentro rango",
    "estado":           "Estado",
})

df_diagnostico_display.style.applymap(colorear_estado, subset=["Estado"])

AttributeError: 'Styler' object has no attribute 'applymap'

In [ ]:
# Variables que requieren atención (Revisar o Ajustar)
atencion = df_diagnostico[df_diagnostico["estado"].isin(["Revisar", "Ajustar"])]
if len(atencion) == 0:
    print("Todas las variables están dentro del rango esperado. No hay ajustes urgentes.")
else:
    print(f"{len(atencion)} variable(s) requieren revisión:\n")
    for _, fila in atencion.iterrows():
        print(
            f"  [{fila['estado']:8s}] {fila['variable']:45s} "
            f"| Media: {fila['media_sintetica']:7.2f} "
            f"| {fila['pct_dentro_rango']:.1f}% dentro de {fila['rango_normal']}"
        )
    print(
        "\nPara corregirlas: abre src/variables.py, ajusta el rango_sintetico"
        " y/o rango_normal de esas variables, y vuelve a ejecutar el notebook 01."
    )

## Sección 8: Tu trabajo — Validación con la literatura

---

### Lo que necesitas hacer

La Sección 3 de este notebook contiene el esqueleto de la tabla comparativa. Para completar la validación de tu TFM, necesitas rellenar los valores de la literatura para **cada variable**. A continuación te indico el proceso paso a paso.

---

### Paso 1: Identificar tu población objetivo

Antes de buscar referencias, define con precisión a quién va dirigido el sistema:
- ¿Qué deporte?
- ¿Qué género?
- ¿Qué rango de edad?
- ¿Nivel competitivo (amateur, semiprofesional, élite)?

Esta definición determinará qué estudios son comparables con tus datos sintéticos.

---

### Paso 2: Buscar valores normativos por variable

Para cada variable, busca en PubMed o SPORTDiscus artículos que reporten:
- Media ± desviación estándar en una muestra comparable
- O percentiles (P25, P50, P75)

Términos de búsqueda útiles:
- `"quadriceps strength" "normative values" athletes`
- `"Y-Balance Test" "reference values" football`
- `"single-leg hop" "limb symmetry index" soccer`
- `"dorsiflexion" "weight-bearing lunge" athletes`

---

### Paso 3: Rellenar la tabla (Sección 3)

En la celda de la Sección 3, sustituye `None` por el valor numérico encontrado:

```python
# Antes (placeholder):
"media_literatura": None,  # TODO ROBERTO

# Después (con valor real):
"media_literatura": 320.0,  # Stark et al. (2011)
"de_literatura":     65.0,
"referencia": "Stark T et al. (2011). Quadriceps muscle strength in young athletes. J Strength Cond Res, 25(7), 1935-1940.",
"poblacion": "Futbolistas masculinos, 20-25 años, nivel semiprofesional (n=42)",
```

Vuelve a ejecutar toda la sección 3 y 4 para ver la comparación actualizada.

---

### Paso 4: Interpretar las diferencias

Si la media sintética difiere de la media de la literatura en **más del 20%**, tienes dos opciones:

**Opción A — Ajustar los datos sintéticos**: Abre `src/variables.py`, modifica `rango_sintetico` y/o el parámetro de la media en el generador, y vuelve a ejecutar el notebook 01. Esto requiere volver a entrenar el modelo (notebooks 03 y 04).

**Opción B — Documentar la diferencia**: Si la diferencia tiene una justificación clínica (por ejemplo, tu población objetivo es más joven o tiene más masa muscular que la del estudio de referencia), documéntalo en el TFM como limitación.

---

### Paso 5: Documentar en el TFM

A continuación tienes una plantilla de párrafo para la **sección de Limitaciones** de tu TFM. Rellena los huecos en corchetes:

---

> **Plantilla para la sección de Limitaciones del TFM:**
>
> *"El presente trabajo utiliza datos sintéticos para el desarrollo y validación técnica del sistema IntApp. Los datos fueron generados mediante un proceso estocástico controlado (notebook 01) con parámetros de distribución derivados de los rangos clínicos reportados en la literatura para [describe tu población: ej. futbolistas masculinos de nivel semiprofesional]. La validación de la plausibilidad de las distribuciones sintéticas (notebook 06) mostró que [X] de las [N] variables presentaban medias dentro del ±20% de los valores de referencia publicados, y que las correlaciones clínicas esperadas —incluyendo la relación negativa entre fuerza de glúteo medio y valgo dinámico de rodilla (r = [valor], p [valor])— fueron reproducidas correctamente en los datos generados.*
>
> *No obstante, el uso de datos sintéticos constituye una limitación metodológica fundamental. La validación clínica del sistema requeriría un estudio prospectivo con datos de pacientes reales, seguimiento longitudinal, y la verificación de que las predicciones del modelo se corresponden con los eventos de lesión observados. El presente trabajo debe entenderse como una demostración de viabilidad metodológica: establece que el enfoque técnico es correcto y que los datos con los que ha sido entrenado el modelo son clínicamente plausibles, pero no valida la capacidad predictiva real del sistema en una cohorte de deportistas.*"

---

### Resumen de tareas pendientes

| Tarea | Estado |
|-------|--------|
| Definir población objetivo del TFM | Pendiente |
| Buscar valores normativos para cada variable | Pendiente |
| Rellenar tabla de referencias (Sección 3) | Pendiente |
| Volver a ejecutar secciones 3 y 4 | Pendiente |
| Interpretar diferencias > 20% | Pendiente |
| Ajustar `src/variables.py` si es necesario | Pendiente |
| Copiar párrafo de limitaciones al TFM | Pendiente |

---

> **Recuerda**: si ajustas `src/variables.py`, debes volver a ejecutar los notebooks en orden: 01 → 02 → 03 → 04 → 06. Los modelos entrenados en el notebook 03 quedarán desactualizados si cambias los datos.